# Export Qwen2.5-3B English-v1 for Ollama

Run on Kaggle with a T4 GPU and Internet enabled, or on Colab T4. Attach the **saved English-v1 training artifact** as an input dataset (a directory containing `model_lock.json`, or `w2w_english_v1_artifacts.zip`). On Colab, mount Drive and place the artifact there. This notebook merges the existing adapter with the exact Hugging Face base revision; it does **not** retrain.

The merged model is about 6 GB. Save the complete output directory as a Kaggle Version output or copy it to Drive before the runtime ends. The next step on your local machine is `tools/import_qwen25_ollama.py`.


## 1. Dependencies

Keep the runtime's CUDA-enabled PyTorch. The export runs in a fresh subprocess after installation, so the notebook kernel does not need a restart.


In [ ]:
%pip -q install 'transformers==4.53.3' 'peft==0.17.1' 'accelerate==1.10.1' 'safetensors==0.6.2' 'huggingface-hub==0.34.4'


## 2. Locate the saved artifact

Set `ARTIFACTS_OVERRIDE` if several matching inputs exist. For a ZIP, extraction happens in the runtime working directory. The source files remain untouched.


In [ ]:
from pathlib import Path
import zipfile, shutil
ON_KAGGLE = Path('/kaggle/working').is_dir()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')
SEARCH = Path('/kaggle/input') if ON_KAGGLE else Path('/content/drive/MyDrive')
ARTIFACTS_OVERRIDE = None  # e.g. Path('/kaggle/input/my-artifacts')
OUTPUT_OVERRIDE = None     # e.g. Path('/content/drive/MyDrive/WhatToWatch/merged')
if ARTIFACTS_OVERRIDE is not None:
    ARTIFACTS = Path(ARTIFACTS_OVERRIDE)
else:
    candidates = sorted(SEARCH.glob('**/model_lock.json'))
    if len(candidates) == 1:
        ARTIFACTS = candidates[0].parent
    else:
        archives = sorted(SEARCH.glob('**/w2w_english_v1_artifacts*.zip'))
        assert not candidates and len(archives) == 1, f'Set ARTIFACTS_OVERRIDE; found {candidates} and {archives}'
        ARTIFACTS = WORK / 'w2w_english_v1_artifacts_unpacked'
        ARTIFACTS.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(ARTIFACTS)
        locks = sorted(ARTIFACTS.glob('**/model_lock.json'))
        assert len(locks) == 1, f'Expected one model_lock.json in {archives[0]}, found {locks}'
        ARTIFACTS = locks[0].parent
OUTPUT = Path(OUTPUT_OVERRIDE) if OUTPUT_OVERRIDE is not None else WORK / 'wtw-qwen25-english-v1-merged'
assert (ARTIFACTS / 'model_lock.json').is_file(), f'Missing model_lock.json under {ARTIFACTS}'
print('Artifact:', ARTIFACTS)
print('Merged output:', OUTPUT)
print('Free disk GiB:', round(shutil.disk_usage(WORK).free / 2**30, 1))
assert shutil.disk_usage(WORK).free >= 15 * 2**30, 'Need at least 15 GiB free disk for base cache and merged weights'


## 3. Embedded export script

This cell creates the exact version of `tools/export_qwen25_merged.py` from branch `feature/chat-ollama-adapter`. It verifies the saved adapter hashes and merges into the locked base revision.


In [ ]:
import json
EXPORT_SCRIPT = WORK / 'export_qwen25_merged.py'
EXPORT_SCRIPT.write_text("\"\"\"Merge the English-v1 PEFT adapter into its exact Hugging Face base checkpoint.\n\nRun on a CUDA machine with enough free VRAM (for example Kaggle T4) and the\nsame Transformers/PEFT stack recorded in the training artifact. The output is\nan ordinary Safetensors model directory for Ollama import, not a QLoRA adapter.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef sha256(path: Path) -> str:\n    with path.open(\"rb\") as handle:\n        return hashlib.file_digest(handle, \"sha256\").hexdigest()\n\n\ndef verify_artifacts(artifacts: Path) -> tuple[dict, dict, Path]:\n    lock = json.loads((artifacts / \"model_lock.json\").read_text(encoding=\"utf-8\"))\n    training = json.loads((artifacts / \"train_complete.json\").read_text(encoding=\"utf-8\"))\n    adapter = artifacts / \"adapter\"\n    if lock[\"model_id\"] != \"Qwen/Qwen2.5-3B-Instruct\" or len(lock[\"revision\"]) != 40:\n        raise ValueError(\"Unexpected base model lock\")\n    if training.get(\"test_used_for_selection\") is not False:\n        raise ValueError(\"Training selection contract is not verified\")\n    for name, expected in training[\"adapter_files\"].items():\n        path = adapter / name\n        if not path.is_file() or sha256(path) != expected:\n            raise ValueError(f\"Adapter file missing or modified: {path}\")\n    adapter_config = json.loads((adapter / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n    if adapter_config.get(\"peft_type\") != \"LORA\" or lock[\"revision\"] not in adapter_config.get(\n        \"base_model_name_or_path\", \"\"\n    ):\n        raise ValueError(\"Adapter does not identify the locked base revision\")\n    return lock, training, adapter\n\n\ndef export(artifacts: Path, output: Path) -> None:\n    lock, training, adapter = verify_artifacts(artifacts)\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"A CUDA GPU is required for this export\")\n    free_bytes, _ = torch.cuda.mem_get_info(0)\n    if free_bytes < 9 * 2**30:\n        raise RuntimeError(\"At least 9 GiB free GPU memory is required before merging\")\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f\"Choose an empty output directory: {output}\")\n    output.mkdir(parents=True, exist_ok=True)\n\n    base = AutoModelForCausalLM.from_pretrained(\n        lock[\"model_id\"], revision=lock[\"revision\"], torch_dtype=torch.float16,\n        device_map={\"\": 0}, attn_implementation=\"sdpa\", trust_remote_code=False,\n    )\n    peft_model = PeftModel.from_pretrained(base, adapter, is_trainable=False)\n    merged = peft_model.merge_and_unload(safe_merge=True)\n    merged.save_pretrained(output, safe_serialization=True, max_shard_size=\"2GB\")\n    AutoTokenizer.from_pretrained(adapter, use_fast=True).save_pretrained(output)\n\n    files = {path.name: sha256(path) for path in sorted(output.iterdir()) if path.is_file()}\n    if not any(name.endswith(\".safetensors\") for name in files):\n        raise RuntimeError(\"Export created no Safetensors weights\")\n    manifest = {\n        \"base_model\": lock[\"model_id\"], \"base_revision\": lock[\"revision\"],\n        \"adapter_sha256\": training[\"adapter_files\"][\"adapter_model.safetensors\"],\n        \"precision\": \"float16 merged weights\", \"files\": files,\n    }\n    (output / \"export_manifest.json\").write_text(\n        json.dumps(manifest, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    print(f\"Merged model exported to {output}\")\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--artifacts\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path)\n    parser.add_argument(\"--verify-only\", action=\"store_true\")\n    args = parser.parse_args()\n    if args.verify_only:\n        lock, _, _ = verify_artifacts(args.artifacts.resolve())\n        print(f\"Verified adapter for {lock['model_id']} at {lock['revision']}\")\n    elif args.output is None:\n        parser.error(\"--output is required unless --verify-only is set\")\n    else:\n        export(args.artifacts.resolve(), args.output.resolve())\n", encoding='utf-8')
print('Created:', EXPORT_SCRIPT)


## 4. Verify and merge

First run may download the base checkpoint. GPU 0 needs at least 9 GiB free VRAM. The output directory must be empty.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-u', str(EXPORT_SCRIPT), '--artifacts', str(ARTIFACTS), '--verify-only'], check=True)
subprocess.run([sys.executable, '-u', str(EXPORT_SCRIPT), '--artifacts', str(ARTIFACTS), '--output', str(OUTPUT)], check=True)


## 5. Verify export and save it

Keep every file listed in `export_manifest.json`; downloading only the `.safetensors` shards is insufficient. On Kaggle, save a notebook Version with outputs or download the whole directory before the session expires.


In [ ]:
import hashlib
manifest = json.loads((OUTPUT / 'export_manifest.json').read_text())
for name, expected in manifest['files'].items():
    with (OUTPUT / name).open('rb') as handle:
        assert hashlib.file_digest(handle, 'sha256').hexdigest() == expected, name
print('Base:', manifest['base_model'], manifest['base_revision'])
print('Verified files:', len(manifest['files']))
print('Output GiB:', round(sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file()) / 2**30, 2))
print('Save this entire directory:', OUTPUT)
